# smSoD on Kaggle

ECGA with multidimensional split-on-demand discretization, NumPy (CPU) and CuPy (GPU).

Before running, in the right panel **Settings**:
* **Accelerator**: GPU T4 x2 or GPU P100
* **Internet**: On (needs a phone-verified Kaggle account)

Please cite: Che-Wei Liang and Tian-Li Yu, *Novel Discretization Scheme for Multidimensional
Split-on-Demand on Real-Valued Optimization with High Multi-modality*, IJCCI 2025.

In [ ]:
!nvidia-smi -L

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.exists("smSoD"):
    !git clone -q https://github.com/rayliangg/smSoD.git
os.chdir("/kaggle/working/smSoD")
!git pull -q

# kaggle has cupy-cuda13x, use cupy-cuda12x + CUDA 12 libs instead
!pip uninstall -y -q cupy cupy-cuda12x cupy-cuda13x > /tmp/pip.log 2>&1
!pip install -q numpy==2.3.3 cupy-cuda12x==13.6.0 fastrlock==0.8.3 nvidia-cuda-nvrtc-cu12==12.9.86 nvidia-cublas-cu12==12.9.1.4 nvidia-curand-cu12==10.3.10.19 >> /tmp/pip.log 2>&1 && echo "install ok" || tail -20 /tmp/pip.log

libs = !python -c "import nvidia.cuda_nvrtc, nvidia.cublas, nvidia.curand, os; print(':'.join(os.path.join(list(m.__path__)[0], 'lib') for m in (nvidia.cuda_nvrtc, nvidia.cublas, nvidia.curand)))"
os.environ["LD_LIBRARY_PATH"] = libs[-1] + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("LD_LIBRARY_PATH =", os.environ["LD_LIBRARY_PATH"])

In [ ]:
!python -c "import cupy as cp; x = cp.random.RandomState(0).random_sample((4, 3)); print('CuPy OK', cp.__version__, 'nvrtc', cp.cuda.nvrtc.getVersion(), float((x @ cp.ones(3)).sum()))"

## Single run

In [ ]:
!python smSoD_numpy.py --m 5 --k 2 --max_gen 500 --verbose 100
!python smSoD_cupy.py  --m 5 --k 2 --max_gen 500 --verbose 100

## NumPy vs CuPy speed

Short runs (200 generations) to see the speed and how many steps went to the GPU.


In [ ]:
for m, k in [(100, 5), (200, 5), (300, 5)]:
    for v in ["numpy", "cupy"]:
        print(v, "m=%d k=%d" % (m, k))
        !python smSoD_{v}.py --m {m} --k {k} --max_gen 200 --seed 1 --output_file '' | tail -1

## Benchmark

`m = 100 200 300`, `k = 2 3 4 5`, 5 runs each, NumPy and CuPy.

The runs are independent, so the benchmark runs one per CPU core at the same time and puts
the CuPy runs on both GPUs (T4 x2). Watch the GPUs with `!nvidia-smi` in another cell.

A Kaggle session stops after 12 hours. For the full grid use **Save Version -> Save & Run All
(Commit)**, it runs in the background and keeps everything in `/kaggle/working` as output.


In [ ]:
!python benchmark.py --m 100 200 300 --k 2 3 4 5 --runs 5

In [ ]:
from IPython.display import Markdown
text = open("README.md").read()
Markdown(text[text.index("## Performance"):])

Copy the results to `/kaggle/working` so they show up in the **Output** tab:

In [ ]:
import shutil
shutil.copy("README.md", "/kaggle/working/README.md")
shutil.copy("results/benchmark.csv", "/kaggle/working/benchmark.csv")